# OpenSearch Serverless: Hybrid Search & Benchmark

This notebook stores video embeddings in **Amazon OpenSearch Serverless** and performs search.

The key differentiator of OpenSearch Serverless is **hybrid search** — combining vector similarity search (semantic) with keyword search (BM25) simultaneously, which is effective for queries mixing visual and textual information.

Topics covered:
1. Vector ingest and speed measurement
2. k-NN search vs hybrid search comparison
3. Search latency benchmark (p50/p95/p99)
4. Results saved to `benchmark_opensearch.json` for comparison in notebook 04

**Prerequisites:** Complete `01_setup_and_embeddings.ipynb`

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
EMBEDDINGS_S3_PREFIX = config["EMBEDDINGS_S3_PREFIX"]
EMBEDDING_DIMENSION = config["EMBEDDING_DIMENSION"]
VIDEO_IDS = config["VIDEO_IDS"]
OPENSEARCH_ENDPOINT = config.get("OPENSEARCH_ENDPOINT", "")

if not OPENSEARCH_ENDPOINT:
    raise ValueError("OPENSEARCH_ENDPOINT is not set in config.json. Please configure it in notebook 01.")

session = boto3.Session()
s3 = session.client("s3", region_name=REGION)

print(f"OpenSearch: {OPENSEARCH_ENDPOINT}")
print(f"   Vector dimension: {EMBEDDING_DIMENSION}, Videos: {len(VIDEO_IDS)}")

### Parameters

| Parameter | Description | Recommended |
|-----------|-------------|-------------|
| `INDEX_NAME` | Vector index name (must match CloudFormation data access policy) | `video-embeddings-index` |
| `INGEST_BATCH_SIZE` | Documents per bulk API call. Larger = fewer API calls | 50-500 |
| `SEARCH_K` | Number of nearest neighbors to return in k-NN search | 5, 10, 20 |
| `HYBRID_VECTOR_WEIGHT` | Vector similarity weight in hybrid search | 0.5-0.9 |
| `HYBRID_TEXT_WEIGHT` | Keyword matching weight in hybrid search | 0.1-0.5 |

In [ ]:
INDEX_NAME = "video-embeddings-index"
INGEST_BATCH_SIZE = 200
SEARCH_K = 5
HYBRID_VECTOR_WEIGHT = 0.7
HYBRID_TEXT_WEIGHT = 0.3

## 1. Connect to OpenSearch Serverless

Connect to the OpenSearch Serverless collection provisioned by CloudFormation. Uses SigV4 authentication.

In [ ]:
host = OPENSEARCH_ENDPOINT.replace("https://", "")
creds = session.get_credentials().get_frozen_credentials()
auth = AWS4Auth(creds.access_key, creds.secret_key, REGION, "aoss", session_token=creds.token)

os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=60
)
print(f"Connected: {host}")

## 2. Ingest Embeddings

Load embeddings from S3 and ingest into OpenSearch. Ingest time is measured for comparison with S3 Vectors in notebook 04.

> **OpenSearch bulk API**: Uses NDJSON format to index multiple documents at once. Batch size is limited by request size (default 100MB), not document count, so batches can be large.

In [ ]:
def load_embeddings_from_s3():
    """Load embeddings from S3 (searches multiple paths, handles Workshop 1 and 4 formats)."""
    vectors, found_files = [], []
    for prefix in [EMBEDDINGS_S3_PREFIX, 'embeddings/videos/', 'embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.json') and '/config.json' not in obj['Key']:
                found_files.append(obj['Key'])
        if found_files: break

    # Skip output.json/manifest.json if consolidated files exist (avoid duplicates)
    consolidated = [f for f in found_files if f.split('/')[-1] not in ('output.json', 'manifest.json')]
    if consolidated:
        found_files = consolidated

    for s3_key in found_files:
        try: data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)['Body'].read().decode())
        except: continue
        segs = data.get('vectors', data.get('data', []))
        vid = data.get('video_id', '')
        if not vid:
            basename = s3_key.split('/')[-1].replace('.json', '')
            if basename == 'output':
                # Try to get video filename from Bedrock output metadata
                import os as _os
                s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                if not s3_uri:
                    s3_uri = data.get('video', {}).get('mediaSource', {}).get('s3Location', {}).get('uri', '')
                if s3_uri:
                    vid = _os.path.splitext(_os.path.basename(s3_uri))[0]
                else:
                    vid = s3_key.split('/')[-3]
            else:
                vid = basename
        for seg in segs:
            if 'embedding' not in seg: continue
            vectors.append({
                'video_id': vid, 'embedding': seg['embedding'],
                'start_sec': seg.get('startSec', seg.get('start_time', 0)),
                'end_sec': seg.get('endSec', seg.get('end_time', 0)),
                'scope': seg.get('embeddingScope', seg.get('embedding_scope', 'clip')),
                'description': f"{vid} video segment"
            })
    return vectors

vectors = load_embeddings_from_s3()
print(f"Loaded {len(vectors)} vectors from S3")

In [ ]:
# Check and recreate index (clean benchmark)
try:
    if os_client.indices.exists(index=INDEX_NAME):
        os_client.indices.delete(index=INDEX_NAME)
        print(f"   Deleted existing index")
        time.sleep(3)
except: pass

os_client.indices.create(index=INDEX_NAME, body={
    "settings": {"index": {"knn": True}},
    "mappings": {"properties": {
        "embedding": {"type": "knn_vector", "dimension": EMBEDDING_DIMENSION,
                      "method": {"engine": "faiss", "name": "hnsw", "space_type": "cosinesimil"}},
        "video_id": {"type": "keyword"}, "start_sec": {"type": "float"},
        "end_sec": {"type": "float"}, "scope": {"type": "keyword"}, "description": {"type": "text"}
    }}
})
print(f"Index '{INDEX_NAME}' created (faiss/hnsw/cosinesimil)")
time.sleep(10)

# Ingest with timing
print(f"   Ingesting: {len(vectors)} vectors, batch_size={INGEST_BATCH_SIZE}...")
ingest_start = time.time()
ingested = 0
for b in range(0, len(vectors), INGEST_BATCH_SIZE):
    batch = vectors[b:b + INGEST_BATCH_SIZE]
    bulk_body = []
    for v in batch:
        bulk_body.append(json.dumps({"index": {"_index": INDEX_NAME}}))
        bulk_body.append(json.dumps(v))
    resp = os_client.bulk(body="\n".join(bulk_body) + "\n")
    if resp.get('errors'):
        for item in resp['items']:
            if 'error' in item.get('index', {}):
                print(f"   Error: {item['index']['error']}")
                break
    else:
        ingested += len(batch)
ingest_time = time.time() - ingest_start

# OpenSearch Serverless has auto-refresh intervals — wait for indexing to complete
print(f"   {ingested} sent ({ingest_time:.2f}s), waiting for indexing...")
for _ in range(6):
    time.sleep(10)
    count = os_client.count(index=INDEX_NAME)["count"]
    if count > 0:
        break
print(f"Ingest complete: {count} indexed, ingest time {ingest_time:.2f}s")

## 3. Hybrid Search

Hands-on with OpenSearch's key feature: **hybrid search**.

| Search Type | How It Works | Strength |
|-------------|-------------|----------|
| **k-NN (vector)** | Finds semantically closest vectors by cosine similarity | Visual/conceptual similarity |
| **BM25 (keyword)** | Calculates query word frequency in documents | Exact keyword matching |
| **Hybrid** | Normalizes and combines both scores with weighted average | Best of both approaches |

OpenSearch uses `normalization-processor` in a search pipeline to normalize different score scales to 0-1 range before combining. This provides more accurate ranking than a simple `bool` query.

> **Difference from S3 Vectors**: S3 Vectors only supports vector similarity search. If keyword search is needed, OpenSearch is the choice.

In [ ]:
# Create search pipeline
pipeline_body = {
    "description": "Normalization pipeline for hybrid search",
    "phase_results_processors": [{
        "normalization-processor": {
            "normalization": {"technique": "min_max"},
            "combination": {"technique": "arithmetic_mean",
                            "parameters": {"weights": [HYBRID_VECTOR_WEIGHT, HYBRID_TEXT_WEIGHT]}}
        }
    }]
}
os_client.http.put("/_search/pipeline/hybrid-pipeline", body=pipeline_body)
print(f"Search pipeline created (vector={HYBRID_VECTOR_WEIGHT}, text={HYBRID_TEXT_WEIGHT})")

In [ ]:
# Text-to-embedding
from botocore.config import Config
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))

def embed_text(query):
    resp = bedrock.invoke_model(
        modelId="us.twelvelabs.marengo-embed-3-0-v1:0",
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(resp["body"].read())["data"][0]["embedding"]

print("Text embedding function ready")

In [ ]:
# ==============================
# Enter your search query
# ==============================
SEARCH_QUERY = "goal scene in a soccer match"  # ← Try different queries!

query_vector = embed_text(SEARCH_QUERY)
print(f"Query: '{SEARCH_QUERY}' -> {len(query_vector)}-dim vector")

In [ ]:
def fmt_hit(hit):
    src = hit['_source']
    start = src.get('start_sec', src.get('start_time', 0))
    end = src.get('end_sec', src.get('end_time', 0))
    scope = src.get('scope', src.get('embedding_scope', '?'))
    return f"score={hit['_score']:.4f} | {src.get('video_id','?')} | {start:.1f}s-{end:.1f}s | {scope}"

# 1) k-NN 검색
t0 = time.time()
knn_resp = os_client.search(index=INDEX_NAME, body={"size": SEARCH_K, "query": {"knn": {"embedding": {"vector": query_vector, "k": SEARCH_K}}}})
knn_latency = (time.time() - t0) * 1000

print(f"🔍 k-NN 검색 (latency={knn_latency:.1f}ms):")
for i, hit in enumerate(knn_resp["hits"]["hits"], 1):
    print(f"  {i}. {fmt_hit(hit)}")

# 2) 하이브리드 검색
hybrid_body = {"size": SEARCH_K, "query": {"hybrid": {"queries": [
    {"knn": {"embedding": {"vector": query_vector, "k": SEARCH_K}}},
    {"match": {"description": SEARCH_QUERY}}
]}}}
t0 = time.time()
hybrid_resp = os_client.search(index=INDEX_NAME, body=hybrid_body, params={"search_pipeline": "hybrid-pipeline"})
hybrid_latency = (time.time() - t0) * 1000

print(f"\n🔍 하이브리드 검색 (latency={hybrid_latency:.1f}ms):")
for i, hit in enumerate(hybrid_resp["hits"]["hits"], 1):
    print(f"  {i}. {fmt_hit(hit)}")

In [ ]:
# Video playback — k-NN vs Hybrid comparison
from IPython.display import display, HTML

def build_video_mapping():
    mapping = {}
    for prefix in ['videos/', 'vectordb-workshop/videos/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.mp4'):
                import os; mapping[os.path.splitext(os.path.basename(obj['Key']))[0]] = obj['Key']
    for prefix in ['embeddings/videos/', 'vectordb-workshop/embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=500)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('output.json'):
                try:
                    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=obj['Key'])['Body'].read().decode())
                    s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                    if s3_uri:
                        parts = obj['Key'].split('/')
                        mapping[parts[2] if len(parts) > 2 else parts[-2]] = s3_uri.replace(f's3://{S3_BUCKET}/', '')
                except: continue
    return mapping

video_mapping = build_video_mapping()
print(f"Video mapping: {len(video_mapping)} videos")
print(f"   Mapped IDs: {list(video_mapping.keys())}")

def play_result(hit_source, label=''):
    vid = hit_source.get('video_id', '')
    start = hit_source.get('start_sec', hit_source.get('start_time', 0))
    key = video_mapping.get(vid)
    if key:
        url = s3.generate_presigned_url('get_object', Params={'Bucket': S3_BUCKET, 'Key': key}, ExpiresIn=3600)
        display(HTML(f'<h4>{label} {vid} ({start:.1f}s~)</h4><video width="480" controls><source src="{url}#t={start}" type="video/mp4"></video>'))
    else:
        print(f"Video not found: '{vid}' (not in mapping)")

# Compare k-NN Top 1 vs Hybrid Top 1
print('\nk-NN vs Hybrid search result comparison:\n')
if knn_resp['hits']['hits']:
    play_result(knn_resp['hits']['hits'][0]['_source'], label='[k-NN #1]')
if hybrid_resp['hits']['hits']:
    play_result(hybrid_resp['hits']['hits'][0]['_source'], label='[Hybrid #1]')

## 4. Search Benchmark

Systematically measure search latency. Results are saved to `benchmark_opensearch.json` for comparison with S3 Vectors in notebook 04.

### Percentile Latency

Average values can be skewed by outliers, so **percentiles** are used to measure actual service quality:

| Metric | Meaning | Use Case |
|--------|---------|----------|
| **p50** (median) | 50% of requests complete within this time | Typical user experience |
| **p95** | 95% of requests complete within this time | Most users' experience |
| **p99** | 99% of requests complete within this time | Worst case (SLA target) |

In [ ]:
BENCHMARK_ITERATIONS = 20
BENCHMARK_QUERY_COUNT = 5

In [ ]:
import numpy as np

query_vectors = [v["embedding"] for v in vectors if v["scope"] == "asset"][:BENCHMARK_QUERY_COUNT]
if not query_vectors: query_vectors = [vectors[0]["embedding"]]

print(f"Benchmark: {len(query_vectors)} queries x {BENCHMARK_ITERATIONS} iterations\n")

benchmark_results = {"service": "OpenSearch Serverless", "vector_count": len(vectors),
                     "ingest_time": ingest_time, "ingest_batch_size": INGEST_BATCH_SIZE, "search": {}}

for k_val in [5, 10, 20]:
    latencies = []
    for _ in range(BENCHMARK_ITERATIONS):
        for qv in query_vectors:
            t0 = time.time()
            os_client.search(index=INDEX_NAME, body={"size": k_val, "query": {"knn": {"embedding": {"vector": qv, "k": k_val}}}})
            latencies.append((time.time() - t0) * 1000)
    latencies.sort()
    n = len(latencies)
    stats = {"p50": latencies[n//2], "p95": latencies[int(n*0.95)], "p99": latencies[int(n*0.99)], "avg": np.mean(latencies)}
    benchmark_results["search"][f"k={k_val}"] = stats
    print(f"  k={k_val:<3} | p50={stats['p50']:.1f}ms | p95={stats['p95']:.1f}ms | p99={stats['p99']:.1f}ms | avg={stats['avg']:.1f}ms")

print("\nBenchmark complete")

In [ ]:
# Save results (used in 04_comparison.ipynb)
with open("benchmark_opensearch.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"benchmark_opensearch.json saved")
print(f"   Ingest: {ingest_time:.2f}s ({len(vectors)} vectors, batch={INGEST_BATCH_SIZE})")
print(f"   Search p50 (k=5): {benchmark_results['search']['k=5']['p50']:.1f}ms")
print(f"\n   Next: Run 03_s3_vectors.ipynb, then 04_comparison.ipynb")